# Medical Triage Agent AI POC
## Google Colab Training Pipeline


In [ ]:
# ============================================================
# CELLULE 0 — Patch BF16 torch + reset environnement (Colab Free T4)
# ============================================================
import os
import gc
import sys

_modules_to_clear = [m for m in sys.modules if m.startswith((
    "torch", "transformers", "peft", "unsloth", "trl", "bitsandbytes", "accelerate"
))]
for m in _modules_to_clear:
    del sys.modules[m]

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

_original_is_bf16_supported = torch.cuda.is_bf16_supported

def _forced_no_bf16(*args, **kwargs):
    return False

torch.cuda.is_bf16_supported = _forced_no_bf16

gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("AUDIT ENVIRONNEMENT GPU")
print("=" * 60)
print("GPU :", torch.cuda.get_device_name(0))
print("VRAM totale :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("BF16 natif (valeur réelle avant patch) :", _original_is_bf16_supported())
print("BF16 natif torch (forcé) :", torch.cuda.is_bf16_supported())
print("=" * 60)

In [ ]:
from __future__ import annotations
import logging

logging.basicConfig(level=logging.INFO)
logger=logging.getLogger('training_pipeline')


# Section 2 — Bootstrap Runtime


In [ ]:
import sys
import subprocess
import os

UPDATED = False

def install(pkg, version=None):
    global UPDATED
    target = f"{pkg}=={version}" if version else pkg
    print(f"📦 Installation : {target}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-U", target],
        check=True
    )
    UPDATED = True

def install_no_deps(pkg, version=None):
    global UPDATED
    target = f"{pkg}=={version}" if version else pkg
    print(f"📦 Installation (--no-deps) : {target}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-U", "--no-deps", target],
        check=True
    )
    UPDATED = True

print("🚀 INSTALLATION DES DÉPENDANCES\n")

# --- 1. Dépendances de base ---
install("accelerate", "1.14.0")
install("peft", "0.19.1")
install("datasets", "4.0.0")
install("bitsandbytes", "0.49.2")
install("huggingface_hub", "1.20.1")
install("wandb", "0.28.0")
install("evaluate", "0.4.6")
install("sentencepiece", "0.2.1")
install("safetensors", "0.8.0")

# --- 2. Unsloth + sa dépendance obligatoire, sans toucher aux versions fixées ci-dessus/ci-dessous ---
install_no_deps("unsloth")
install_no_deps("unsloth_zoo")

# --- 3. Réinstallation EN DERNIER des packages critiques pour forcer les bonnes versions ---
install("torch", "2.11.0+cu128")
install("transformers", "5.12.1")
install("trl", "1.7.0")
install("torchao", "0.17.0")

print("\n✅ INSTALLATION TERMINÉE")
if UPDATED:
    print("🔄 REDÉMARRAGE OBLIGATOIRE DU RUNTIME")
    os.kill(os.getpid(), 9)

In [ ]:
# ============================================================
# CELLULE 2 — Patch BF16 (torch + unsloth) + audit
# À exécuter juste après le redémarrage automatique du kernel
# (déclenché par la cellule d'installation précédente)
# ============================================================
import gc
import sys
import inspect

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

def _forced_no_bf16(*args, **kwargs):
    return False

# --- 1. Patch torch ---
_original_is_bf16_supported = torch.cuda.is_bf16_supported
torch.cuda.is_bf16_supported = _forced_no_bf16

# --- 2. Import unsloth (maintenant installé, kernel propre) ---
import unsloth

# --- 3. Inspection avant patch ---
print("=" * 60)
print("INSPECTION UNSLOTH")
print("=" * 60)
try:
    print("Valeur actuelle :", unsloth.is_bfloat16_supported())
    print("Fichier source :", inspect.getsourcefile(unsloth.is_bfloat16_supported))
    print("Module :", unsloth.is_bfloat16_supported.__module__)
except Exception as e:
    print("⚠️  Inspection impossible :", e)

# --- 4. Patch unsloth sur toutes les cibles probables ---
_patched_targets = []

try:
    unsloth.is_bfloat16_supported = _forced_no_bf16
    _patched_targets.append("unsloth.is_bfloat16_supported")
except Exception:
    pass

try:
    import unsloth_zoo.utils as uz_utils
    uz_utils.is_bfloat16_supported = _forced_no_bf16
    _patched_targets.append("unsloth_zoo.utils.is_bfloat16_supported")
except (ImportError, AttributeError):
    pass

try:
    import unsloth_zoo
    if hasattr(unsloth_zoo, "is_bfloat16_supported"):
        unsloth_zoo.is_bfloat16_supported = _forced_no_bf16
        _patched_targets.append("unsloth_zoo.is_bfloat16_supported")
except ImportError:
    pass

try:
    _origin_module = sys.modules.get(unsloth.is_bfloat16_supported.__module__)
    if _origin_module is not None:
        setattr(_origin_module, "is_bfloat16_supported", _forced_no_bf16)
        _patched_targets.append(f"{unsloth.is_bfloat16_supported.__module__}.is_bfloat16_supported")
except Exception:
    pass

print("Cibles patchées :", _patched_targets if _patched_targets else "AUCUNE")

# --- 5. Nettoyage mémoire GPU ---
gc.collect()
torch.cuda.empty_cache()

# --- 6. Audit final ---
print("=" * 60)
print("AUDIT ENVIRONNEMENT GPU")
print("=" * 60)
print("GPU :", torch.cuda.get_device_name(0))
print("VRAM totale :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("BF16 natif (valeur réelle) :", _original_is_bf16_supported())
print("BF16 natif torch (forcé) :", torch.cuda.is_bf16_supported())
print("BF16 natif unsloth (forcé) :", unsloth.is_bfloat16_supported())

ok_torch = torch.cuda.is_bf16_supported() == False
ok_unsloth = unsloth.is_bfloat16_supported() == False

if ok_torch and ok_unsloth:
    print("✅ Patchs actifs : torch ET unsloth retournent False")
else:
    print(f"⚠️  ALERTE — torch OK: {ok_torch} / unsloth OK: {ok_unsloth}")
print("=" * 60)

In [ ]:
# =======================================================
# Cellule 2 : 👉 À exécuter APRÈS redémarrage du runtime
# =======================================================

import torch
import transformers
import trl
import accelerate
import peft
import platform

print("=== ENVIRONMENT CHECK ===\n")

print("Python       :", platform.python_version())
print("Torch        :", torch.__version__)
print("Transformers :", transformers.__version__)
print("TRL          :", trl.__version__)
print("Accelerate   :", accelerate.__version__)
print("PEFT         :", peft.__version__)

print("\nCUDA :", torch.version.cuda)
print("GPU  :", torch.cuda.is_available())

print("\nTorch file   :", torch.__file__)
print("Transformers :", transformers.__file__)
print("TRL file     :", trl.__file__)

print("\n=== TORCH OPS CHECK ===")

print("Has _c10d_functional:", hasattr(torch.ops, "_c10d_functional"))

if hasattr(torch.ops, "_c10d_functional"):
    print("_wrap_tensor_autograd:",
          "_wrap_tensor_autograd" in dir(torch.ops._c10d_functional))

In [ ]:
# pip show unsloth

In [ ]:
# =======================================================
# Cellule 3 : 👉 À exécuter uniquement si Cellule 2 OK
# =======================================================

from trl import DPOConfig, DPOTrainer

print("✅ TRL import OK")

In [ ]:
# =======================================================
# Cellule 4 : 👉 À exécuter uniquement si Cellule 2 OK
# =======================================================

from transformers import TrainingArguments

print("✅ TrainingArguments import OK")

In [ ]:
!git clone https://github.com/raym648/medical-triage-agent-ai-poc.git

In [ ]:
%cd /content/medical-triage-agent-ai-poc

In [ ]:
!ls

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT=Path.cwd().resolve()
while PROJECT_ROOT!=PROJECT_ROOT.parent:
    if (PROJECT_ROOT/'backend').exists() and (PROJECT_ROOT/'frontend').exists():
        break
    PROJECT_ROOT=PROJECT_ROOT.parent
else:
    raise RuntimeError('Repository root not found.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))


In [ ]:
required=['backend/app/training/colab','backend/app/training/sft','backend/app/training/dpo','backend/app/training/evaluation']
missing=[p for p in required if not (PROJECT_ROOT/Path(p)).exists()]
if missing:
    raise FileNotFoundError(missing)


In [ ]:
# !grep -n "def get_training_arguments_precision\|def is_bf16_supported\|def is_fp16_supported" -A 15 backend/app/training/colab/colab_environment.py

In [ ]:
# !grep -n "dtype=None\|FastLanguageModel.from_pretrained" -B 3 -A 10 backend/app/training/shared/training_model_loader.py

In [ ]:
# print(model.dtype)  # doit afficher torch.float16 sur T4, plus jamais bfloat16

# Section 3


In [ ]:
import os
import sys

REPO_PATH = "/content/medical-triage-agent-ai-poc"
BACKEND_PATH = os.path.join(REPO_PATH, "backend")

for p in (REPO_PATH, BACKEND_PATH):
    if p not in sys.path:
        sys.path.insert(0, p)

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN_06")
if hf_token:
    os.environ["HF_TOKEN_06"] = hf_token

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

from backend.app.training.colab.colab_environment import log_environment_info
from backend.app.training.colab.colab_gpu_detector import log_gpu_info
from backend.app.training.colab.colab_hf_login import ensure_hf_login
from backend.app.training.colab.colab_checkpoint_sync import (
    create_default_checkpoint_sync,
)

environment = log_environment_info()
gpu_info = log_gpu_info()
hf_status = ensure_hf_login()
print("✅ HF Login :", hf_status)

In [ ]:

# ============================================================
# NOUVEAU — Lecture de retraining_request.json (phase d'information,
# validée à l'étape 6 des échanges de conception).
#
# Le notebook reste le point d'entrée MANUEL et UNIQUE des
# entraînements SFT/DPO/évaluation clinique (aucun lancement
# automatique de session Colab). Cette cellule se contente d'afficher
# un résumé de la demande de réentraînement éventuellement publiée
# par le workflow GitHub Actions "retraining-detection.yml", pour
# que la décision finale de lancer (ou non) l'entraînement reste
# entièrement humaine.
#
# Aucune modification des cellules suivantes (Section 4/5/6 —
# runpy.run_module sur train_sft / train_dpo / clinical_eval_runner) :
# cette cellule est purement informative et n'interrompt jamais
# l'exécution du notebook, même en cas d'erreur réseau ou d'absence
# de demande.
#
# Le fichier est stocké sur le même repo HF que les checkpoints
# (RemDev-AI/medical-triage-agent-ai-poc-models), déjà utilisé par
# ColabCheckpointSync — aucun nouveau repo, aucun nouveau token requis.
# ============================================================

import os
import sys

import json
from huggingface_hub import hf_hub_download, HfApi

RETRAINING_REQUEST_REPO_ID = "RemDev-AI/medical-triage-agent-ai-poc-models"
RETRAINING_REQUEST_REMOTE_PATH = "retraining/retraining_request.json"

retraining_request = None

try:
    request_local_path = hf_hub_download(
        repo_id=RETRAINING_REQUEST_REPO_ID,
        repo_type="model",
        filename=RETRAINING_REQUEST_REMOTE_PATH,
        revision="main",
    )
    with open(request_local_path, "r", encoding="utf-8") as handle:
        retraining_request = json.load(handle)
except Exception as exc:
    print(
        "ℹ️ Aucune demande de réentraînement lisible sur Hugging Face "
        f"({RETRAINING_REQUEST_REPO_ID}/{RETRAINING_REQUEST_REMOTE_PATH}) "
        f"— détail : {exc}"
    )

print("=" * 60)
print("DEMANDE DE RÉENTRAÎNEMENT")
print("=" * 60)

if retraining_request is None:
    print("Aucune demande de réentraînement détectée.")
    print("Vous pouvez tout de même lancer un entraînement manuel ci-dessous.")
elif retraining_request.get("status") != "pending":
    print(
        f"Dernière demande connue déjà traitée (status="
        f"{retraining_request.get('status')!r}, "
        f"consumed_at={retraining_request.get('notebook', {}).get('consumed_at')})."
    )
    print("Aucune nouvelle demande en attente.")
else:
    reason = retraining_request.get("reason", {})
    thresholds = retraining_request.get("thresholds", {})
    print("✔ Demande de réentraînement détectée (status=pending)")
    print(f"  Créée le        : {retraining_request.get('created_at')}")
    print(f"  Run GitHub      : {retraining_request.get('run_id')}")
    print(
        f"  Cas nécessitant une revue humaine : "
        f"{reason.get('requires_human_review_count')} "
        f"/ {reason.get('total_clinical_entries_analyzed')} analysés "
        f"(seuil : {thresholds.get('min_disagreements')})"
    )
    print(
        f"  Taux d'erreur API : {reason.get('error_rate_percent')}% "
        f"(seuil : {thresholds.get('max_error_rate_percent')}%)"
    )
    print()
    print("La décision de lancer l'entraînement reste manuelle : ")
    print("exécutez les cellules suivantes (Section 4/5/6) si vous validez ce résumé.")

# Best-effort : marque la demande comme "consommée" dès sa lecture par
# le notebook, sans jamais interrompre le pipeline en cas d'échec
# (cf. commentaire ci-dessus — cette étape reste purement informative).
# La clôture complète (status="completed"/"failed") est laissée à
# l'Étape 4 (clôture automatique), traitée séparément et non couverte
# par cette cellule.
if retraining_request is not None and retraining_request.get("status") == "pending":
    try:
        from datetime import datetime, timezone
        import tempfile

        retraining_request["notebook"]["consumed_at"] = datetime.now(
            timezone.utc
        ).isoformat()
        retraining_request["notebook"]["consumed_by"] = "colab-notebook"

        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".json", delete=False, encoding="utf-8"
        ) as tmp:
            json.dump(retraining_request, tmp, indent=2, ensure_ascii=False)
            tmp_path = tmp.name

        HfApi().upload_file(
            path_or_fileobj=tmp_path,
            path_in_repo=RETRAINING_REQUEST_REMOTE_PATH,
            repo_id=RETRAINING_REQUEST_REPO_ID,
            repo_type="model",
            commit_message="Retraining request marked as consumed by notebook",
        )
    except Exception as exc:
        print(f"⚠️ Impossible de marquer la demande comme consommée : {exc}")


# Section 4


In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')  # ← ligne ajoutée

import runpy
runpy.run_module('backend.app.training.sft.train_sft',run_name='__main__')


# Section 5


In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')  # ← ligne ajoutée

import runpy
runpy.run_module('backend.app.training.dpo.train_dpo',run_name='__main__')


# Section 6


In [ ]:

import os
from google.colab import userdata

# try:
#     from google.colab import drive
#     drive.mount("/content/drive")
# except Exception:
#     pass

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['CLINICAL_EVAL_SAMPLE_SIZE'] = '40'  # ← ligne ajoutée : échantillon réduit

import runpy
runpy.run_module('backend.app.training.evaluation.clinical_eval_runner', run_name='__main__')


# Section 7 — Clôture automatique de la demande de réentraînement

In [ ]:

# ============================================================
# NOUVELLE CELLULE — Clôture automatique de la demande de
# réentraînement (Étape 4 du schéma validé).
#
# À insérer APRÈS la cellule "Section 6" (clinical_eval_runner,
# runpy.run_module('backend.app.training.evaluation.clinical_eval_runner')),
# et AVANT la cellule "Section 8" existante (diagnostic Hub).
#
# Aucune cellule existante n'est modifiée : ceci est une cellule
# ENTIÈREMENT NOUVELLE. Elle réutilise la variable globale
# `retraining_request` déjà lue et marquée "consommée" par la
# cellule 19 modifiée — Colab exécute toutes les cellules dans le
# même kernel/namespace, donc cette variable Python reste disponible
# sans nouveau téléchargement.
#
# CORRECTIF (suite à lecture complète de clinical_eval_runner.py et
# evaluation_report.py) — cette version N'APPROXIME PLUS RIEN :
#   - model_name = dpo_config["tracking"]["wandb_run_name"], lu
#     directement depuis backend/app/training/dpo/dpo_config_validation.yaml
#     déjà cloné localement dans le repo (cf. cellule 10, git clone).
#     C'est exactement la même valeur que celle utilisée par
#     clinical_eval_runner.py::__main__ pour appeler evaluate_model().
#   - Chemin remote exact (cf. push_evaluation_reports_to_huggingface,
#     training_type="dpo" par défaut) :
#       dpo/evaluation_reports/{model_name}/evaluation_report.json
#   - Statut canonique : report_data["overall_status"] == "PASS"
#     (calculé par clinical_gate_status(), cf. evaluate_model()),
#     PAS "thresholds_passed" qui n'existe qu'imbriqué sous
#     report_data["safety"]["thresholds_passed"] (complément, pas
#     source de vérité).
#
# Comme la phase d'information (cellule 19), cette cellule est
# purement best-effort : elle ne doit JAMAIS interrompre le
# notebook, même si le réseau échoue ou si le rapport est introuvable
# (ex: hf_hub.enabled=false dans evaluation_config.yaml pour ce run).
# ============================================================

import json
import tempfile
import yaml
from datetime import datetime, timezone

from huggingface_hub import HfApi, hf_hub_download

HF_MODELS_REPO_ID = "RemDev-AI/medical-triage-agent-ai-poc-models"
RETRAINING_REQUEST_REMOTE_PATH = "retraining/retraining_request.json"
DPO_CONFIG_LOCAL_PATH = (
    f"{BACKEND_PATH}/app/training/dpo/dpo_config_validation.yaml"
)

closure_status = "failed"
closure_details = {}

try:
    with open(DPO_CONFIG_LOCAL_PATH, "r", encoding="utf-8") as handle:
        dpo_config = yaml.safe_load(handle)

    model_name = dpo_config["tracking"]["wandb_run_name"]

    remote_report_path = f"dpo/evaluation_reports/{model_name}/evaluation_report.json"

    report_local_path = hf_hub_download(
        repo_id=HF_MODELS_REPO_ID,
        repo_type="model",
        filename=remote_report_path,
        revision="main",
    )

    with open(report_local_path, "r", encoding="utf-8") as handle:
        report_data = json.load(handle)

    overall_status = report_data.get("overall_status", "UNKNOWN")
    safety_thresholds_passed = report_data.get("safety", {}).get("thresholds_passed")

    closure_status = "completed" if overall_status == "PASS" else "failed"
    closure_details = {
        "evaluation_report_path": remote_report_path,
        "overall_status": overall_status,
        "safety_thresholds_passed": safety_thresholds_passed,
    }

    print(f"✔ Rapport d'évaluation utilisé : {remote_report_path}")
    print(f"  overall_status = {overall_status}")
    print(f"  safety.thresholds_passed = {safety_thresholds_passed}")

except FileNotFoundError as exc:
    print(
        "⚠️ Rapport d'évaluation introuvable sur Hugging Face — "
        "vérifier evaluation_config.yaml (hf_hub.enabled / "
        f"attach_evaluation_reports). Détail : {exc}"
    )
    print("Clôture en status='failed' par précaution.")
except Exception as exc:
    print(f"⚠️ Impossible de déterminer le résultat de l'évaluation : {exc}")
    print("Clôture en status='failed' par précaution.")

# ------------------------------------------------------------
# Référence du checkpoint DPO final utilisé/produit par ce run.
# ------------------------------------------------------------
checkpoint_reference = None
try:
    dpo_final_files = [
        f for f in HfApi().list_repo_files(repo_id=HF_MODELS_REPO_ID, repo_type="model")
        if f.startswith("dpo/final/")
    ]
    if dpo_final_files:
        checkpoint_reference = "dpo/final"
except Exception as exc:
    print(f"⚠️ Impossible de vérifier la présence de dpo/final/ : {exc}")

# ------------------------------------------------------------
# Mise à jour et publication de retraining_request.json
# ------------------------------------------------------------
if retraining_request is not None:
    retraining_request["status"] = closure_status
    retraining_request["closure"] = {
        "closed_at": datetime.now(timezone.utc).isoformat(),
        "closed_by": "colab-notebook",
        "checkpoint_reference": checkpoint_reference,
        **closure_details,
    }

    try:
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".json", delete=False, encoding="utf-8"
        ) as tmp:
            json.dump(retraining_request, tmp, indent=2, ensure_ascii=False)
            tmp_path = tmp.name

        HfApi().upload_file(
            path_or_fileobj=tmp_path,
            path_in_repo=RETRAINING_REQUEST_REMOTE_PATH,
            repo_id=HF_MODELS_REPO_ID,
            repo_type="model",
            commit_message=f"Retraining request closed (status={closure_status})",
        )
        print(f"✔ retraining_request.json clôturé (status={closure_status}).")
    except Exception as exc:
        print(f"⚠️ Impossible de publier la clôture sur Hugging Face : {exc}")
else:
    print(
        "ℹ️ Aucune demande de réentraînement n'était en attente au début du "
        "run — rien à clôturer."
    )


# Section 8


In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

from backend.app.training.colab.colab_checkpoint_sync import create_default_checkpoint_sync

for training_type in ("sft", "dpo"):

    print(f"\n{'=' * 60}")
    print(f"DIAGNOSTIC HUB : {training_type.upper()}")
    print(f"{'=' * 60}")

    checkpoint_sync = create_default_checkpoint_sync(
        output_dir=f"./outputs/{training_type}",
        training_type=training_type,
    )

    # Tente de restaurer le dernier checkpoint depuis Hugging Face.
    # Retourne None si aucun checkpoint n'existe pour ce training_type.
    restored_path = checkpoint_sync.restore_latest_checkpoint_from_huggingface()

    if restored_path is None:
        print(f"❌ Aucun checkpoint {training_type} trouvé sur le Hub "
              f"({checkpoint_sync.hf_repo_id}).")
        continue

    from pathlib import Path
    path = Path(restored_path)

    if not path.exists():
        print(f"⚠️ Chemin retourné introuvable sur le disque : {path}")
        continue

    files = sorted(
        str(f.relative_to(path)) for f in path.rglob("*") if f.is_file()
    )
    total_size = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())

    print(f"✅ Dernier checkpoint {training_type} restauré : {path}")
    print(f"   {len(files)} fichier(s), {total_size / 1024:.1f} KB")
    for f in files:
        print(f"     - {f}")
